# EDA + Baseline: AI4I 2020 Predictive Maintenance

This notebook walks through:
1. Loading + sanity-checking the data
2. Class imbalance (the headline finding)
3. Feature distributions split by target
4. Baseline LogisticRegression
5. Comparison to the tuned GradientBoosting from `src.train`

Run this from the project root so the `src` package imports work.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data import load_dataframe, split, NUMERIC_FEATURES, TARGET
from src.features import build_preprocessor

sns.set_theme(style="whitegrid")

df = load_dataframe(Path('../data/ai4i2020.csv'))
df.head()

In [ ]:
print(f"Shape: {df.shape}")
print(f"Positive rate: {df[TARGET].mean():.3%}")
df[TARGET].value_counts().plot(kind='bar', title='Class distribution');

**Key finding:** ~3.4% of rows are failures. Accuracy is a useless metric here — a model that always predicts 'no failure' would score 96.6%.
We'll use **PR-AUC** as our primary metric and tune the decision threshold separately.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, NUMERIC_FEATURES):
    sns.kdeplot(data=df, x=col, hue=TARGET, ax=ax, common_norm=False)
    ax.set_title(col)
axes.flat[-1].axis('off')
fig.suptitle('Feature distributions by failure class', y=1.02)
fig.tight_layout();

**Observations:** torque and tool-wear distributions visibly shift between classes — likely strong predictors. Rotational speed has a more subtle shift.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score

X_train, X_test, y_train, y_test = split(df)

baseline = Pipeline([
    ('preprocess', build_preprocessor()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
baseline.fit(X_train, y_train)

y_proba = baseline.predict_proba(X_test)[:, 1]
print(f"Baseline LogReg — ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"Baseline LogReg — PR-AUC : {average_precision_score(y_test, y_proba):.4f}")

Now compare against the tuned model trained by `python -m src.train` — load `metrics.json` and inspect.

In [ ]:
import json
metrics_path = Path('../metrics.json')
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
else:
    print('Run `python -m src.train` from the project root first.')